# Publication notes

This is the original exploratory workflow with saved outputs removed and paths made configurable. Collection and analytical logic is retained. Source websites and identifiers may change; notebooks 01–02 are an inspected research workflow, not a guaranteed unattended scraper. Review collection permissions and source-specific settings before recollection. Do not run against your only copy of historical data: original export cells can overwrite files.

Run from a clean kernel in order. Data is local and excluded from GitHub. Historical saved review text and reviewer details are not published as notebook outputs.


In [ ]:
from pathlib import Path
import os

# Configure KBEAUTY_PROJECT_ROOT to keep data outside the cloned repository.
_working_dir = Path.cwd().resolve()
_default_root = _working_dir.parent if _working_dir.name == "notebooks" else _working_dir
PROJECT_ROOT = Path(os.environ.get("KBEAUTY_PROJECT_ROOT", str(_default_root))).expanduser().resolve()
# Original collection cells use relative data/ paths.
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)


In [ ]:
from pathlib import Path
import pandas as pd

data_folder = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "all_products"
)

tables = {}

for name in [
    "products",
    "reviews",
    "ingredients",
    "review_skin_concerns"
]:
    file_path = data_folder / f"{name}.csv"

    tables[name] = pd.read_csv(
        file_path,
        dtype={
            "product_id": str,
            "review_id": str,
            "sku": str
        }
    )

    print(f"{name}: {len(tables[name]):,} rows loaded")

products = tables["products"]
reviews = tables["reviews"]
ingredients = tables["ingredients"]
concerns = tables["review_skin_concerns"]

In [ ]:
# Add brand labels without changing the original reviews table.
audit_reviews = reviews.merge(
    products[["product_id", "brand"]],
    on="product_id",
    how="left",
    validate="many_to_one"
)

assert audit_reviews["brand"].notna().all(), "Unmatched product IDs."

def has_value(series):
    return (
        series.notna()
        & series.astype("string").str.strip().ne("").fillna(False)
    )

coverage = (
    audit_reviews.groupby("brand")
    .size()
    .rename("total_reviews")
    .to_frame()
)

for field in ["age", "skin_type", "skincare_goal"]:
    if field in audit_reviews.columns:
        coverage[f"{field}_reported"] = (
            has_value(audit_reviews[field])
            .groupby(audit_reviews["brand"])
            .sum()
        )

# Count reviews with concerns, rather than individual concern pairs.
coverage["skin_concern_reported"] = (
    audit_reviews["review_id"].isin(concerns["review_id"])
    .groupby(audit_reviews["brand"])
    .sum()
)

for column in list(coverage.columns):
    if column.endswith("_reported"):
        coverage[column.replace("_reported", "_coverage_pct")] = (
            100 * coverage[column] / coverage["total_reviews"]
        ).round(1)

print("REVIEWER INFORMATION COVERAGE")
print(coverage.to_string())

print("\nSKIN-CONCERN LABELS")
print(
    concerns["skin_concern"]
    .value_counts()
    .rename_axis("skin_concern")
    .to_string()
)

print("\nSKIN-TYPE LABELS")
print(reviews["skin_type"].value_counts(dropna=False).to_string())

In [ ]:
# Work on copies; preserve the original columns and saved files.
reviews_clean = reviews.copy()
concerns_clean = concerns.copy()

# 1. Standardize skin-type spacing and capitalization.
skin_type_map = {
    "combination": "Combination",
    "normal": "Normal",
    "dry": "Dry",
    "oily": "Oily",
    "sensitive": "Sensitive"
}

skin_type_keys = (
    reviews_clean["skin_type"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

unknown_types = set(skin_type_keys.dropna()) - set(skin_type_map)
assert not unknown_types, f"Unmapped skin types: {unknown_types}"

reviews_clean["skin_type_standardized"] = skin_type_keys.map(
    skin_type_map
)

# 2. Standardize concern labels conservatively.
concern_map = {
    "dryness": "Dryness",
    "ageing": "Aging",
    "aging": "Aging",
    "acne": "Acne",
    "sensitivity": "Sensitivity",
    "sensitive": "Sensitivity",
    "pigmentation": "Pigmentation",
    "wrinkles": "Wrinkles",
    "large pores": "Large pores",
    "sun damage": "Sun damage",
    "oiliness": "Oiliness",
    "redness": "Redness",
    "blackheads": "Blackheads",
    "dehydrated": "Dehydration",
    "dullness": "Dullness",
    "uneven skin tone": "Uneven skin tone",
    "pore congestion": "Pore congestion",
    "scarring": "Scarring",
    "others": "Other",
    "anti-ageing": "Anti-aging",
    "loss of elasticity": "Loss of elasticity",
    "acne scarring": "Acne scarring"
}

concern_keys = (
    concerns_clean["skin_concern"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

unknown_concerns = set(concern_keys.dropna()) - set(concern_map)
assert not unknown_concerns, (
    f"Unmapped skin concerns: {unknown_concerns}"
)

concerns_clean["skin_concern_standardized"] = concern_keys.map(
    concern_map
)

# Count each review once per standardized concern.
# Keep concerns_clean intact so all original labels remain available.
concern_pairs = concerns_clean[
    ["review_id", "skin_concern_standardized"]
].drop_duplicates()

assert len(reviews_clean) == len(reviews)
assert len(concerns_clean) == len(concerns)

print("STANDARDIZED SKIN TYPES")
print(
    reviews_clean["skin_type_standardized"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nSTANDARDIZED CONCERNS — UNIQUE REVIEWS")
print(
    concern_pairs["skin_concern_standardized"]
    .value_counts()
    .to_string()
)

print("\nOriginal concern rows:", len(concerns_clean))
print("Unique standardized review–concern pairs:", len(concern_pairs))

In [ ]:
# Inspect possible formatting differences without changing source names.
ingredients_audit = ingredients.copy()

ingredients_audit["name_key"] = (
    ingredients_audit["ingredient_name"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace(r"\s*/\s*", "/", regex=True)
    .str.casefold()
)

assert ingredients_audit["name_key"].notna().all()
assert ingredients_audit["name_key"].ne("").all()

# Find original spellings that share the same formatting-normalized key.
name_variants = (
    ingredients_audit.groupby("name_key")["ingredient_name"]
    .agg(lambda values: sorted(set(values)))
)

print("INGREDIENT NAME COUNTS")
print("Ingredient rows:", len(ingredients_audit))
print("Distinct original names:", ingredients["ingredient_name"].nunique())
print("Distinct names after formatting cleanup:",
      ingredients_audit["name_key"].nunique())

print("\nFORMATTING VARIANTS")
for key, names in name_variants.items():
    if len(names) > 1:
        print(f"{key}: {names}")

print("\nALL INGREDIENT NAME KEYS")
for key in sorted(ingredients_audit["name_key"].unique()):
    print(key)

In [ ]:
ingredients_clean = ingredients.copy()

# Normalize formatting only.
ingredients_clean["ingredient_name_standardized"] = (
    ingredients_clean["ingredient_name"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace(r"\s*/\s*", "/", regex=True)
    .str.casefold()
)

# Consolidate the multilingual water label.
ingredient_aliases = {
    "water/aqua/eau": "water"
}

ingredients_clean["ingredient_name_standardized"] = (
    ingredients_clean["ingredient_name_standardized"]
    .replace(ingredient_aliases)
)

# Preserve source rows and ingredient order.
assert len(ingredients_clean) == len(ingredients)
assert ingredients_clean["ingredient_name"].equals(
    ingredients["ingredient_name"]
)
assert ingredients_clean["ingredient_position"].equals(
    ingredients["ingredient_position"]
)
assert ingredients_clean["ingredient_name_standardized"].notna().all()
assert ingredients_clean["ingredient_name_standardized"].ne("").all()

# Check whether normalization created repeated ingredients within a product.
duplicate_mask = ingredients_clean.duplicated(
    ["product_id", "ingredient_name_standardized"],
    keep=False
)

print("Ingredient rows:", len(ingredients_clean))
print(
    "Distinct standardized ingredients:",
    ingredients_clean["ingredient_name_standardized"].nunique()
)
print("Rows in repeated product–ingredient pairs:", duplicate_mask.sum())

if duplicate_mask.any():
    print(
        ingredients_clean.loc[
            duplicate_mask,
            [
                "product_id",
                "ingredient_position",
                "ingredient_name",
                "ingredient_name_standardized"
            ]
        ].to_string(index=False)
    )

# Count products containing each ingredient—not ingredient-row frequency.
ingredient_product_counts = (
    ingredients_clean.groupby("ingredient_name_standardized")["product_id"]
    .nunique()
    .sort_values(ascending=False)
)

print("\nMOST COMMON INGREDIENTS — NUMBER OF PRODUCTS")
print(ingredient_product_counts.head(20).to_string())

In [ ]:
audit = reviews_clean.merge(
    products[["product_id", "brand"]],
    on="product_id",
    how="left",
    validate="many_to_one"
)

print("AGE LABELS BY BRAND")
print(
    audit.groupby("brand")["age"]
    .value_counts(dropna=False)
    .to_string()
)

# Display the original values, including missing information.
for field in ["verified", "incentivized", "sponsored", "incentive_type"]:
    if field in audit.columns:
        print(f"\n{field.upper()} — BY BRAND")
        print(
            audit.groupby("brand")[field]
            .value_counts(dropna=False)
            .to_string()
        )

print("\nREVIEW LANGUAGES BY BRAND")
print(
    audit.groupby("brand")["language"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nMISSING OR BLANK REVIEW TEXT BY BRAND")
missing_text = (
    audit["review_text"].isna()
    | audit["review_text"]
        .astype("string")
        .str.strip()
        .eq("")
        .fillna(False)
)

print(
    missing_text.groupby(audit["brand"])
    .sum()
    .to_string()
)

In [ ]:
# Keep missing boolean values as unknown, rather than False.
def to_nullable_boolean(series):
    keys = series.astype("string").str.strip().str.casefold()

    mapping = {
        "true": True,
        "false": False,
        "yes": True,
        "no": False
    }

    unknown = set(keys.dropna()) - set(mapping)
    if unknown:
        raise ValueError(f"Unexpected boolean labels: {unknown}")

    return keys.map(mapping).astype("boolean")


for field in ["verified", "incentivized", "sponsored"]:
    reviews_clean[f"{field}_standardized"] = to_nullable_boolean(
        reviews_clean[field]
    )

# Use a broader 55+ category for age comparisons.
# The original age column retains each provider's exact label.
age_map = {
    "Under 18": "Under 18",
    "18-24": "18-24",
    "25-34": "25-34",
    "35-44": "35-44",
    "45-54": "45-54",
    "55-64": "55+",
    "Over 65": "55+",
    "65+": "55+",
    "55+": "55+"
}

age_keys = reviews_clean["age"].astype("string").str.strip()

unknown_ages = set(age_keys.dropna()) - set(age_map)
assert not unknown_ages, f"Unmapped age labels: {unknown_ages}"

reviews_clean["age_group_analysis"] = age_keys.map(age_map)

# Preserve all languages and flag English according to source metadata.
reviews_clean["language_standardized"] = (
    reviews_clean["language"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

reviews_clean["is_english"] = (
    reviews_clean["language_standardized"]
    .eq("en")
    .astype("boolean")
)

assert len(reviews_clean) == 3600

print("Reviews retained:", len(reviews_clean))

print("\nAGE GROUPS FOR ANALYSIS")
print(
    reviews_clean["age_group_analysis"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nENGLISH LANGUAGE FLAG")
print(reviews_clean["is_english"].value_counts(dropna=False).to_string())

print("\nINCENTIVIZED FLAG")
print(
    reviews_clean["incentivized_standardized"]
    .value_counts(dropna=False)
    .to_string()
)

In [ ]:
from pathlib import Path

# Assemble the cleaned tables.
products_clean = products.copy()

products_clean["analysis_segment"] = products_clean["segment"].replace({
    "Toner / Essence": "Toner"
})

cleaned_tables = {
    "products": products_clean,
    "reviews": reviews_clean,
    "ingredients": ingredients_clean,
    "review_skin_concerns": concerns_clean
}

# Validate counts and relationships before saving.
assert len(products_clean) == 12
assert len(reviews_clean) == 3600
assert len(ingredients_clean) == 346
assert len(concerns_clean) == 4277

assert products_clean["product_id"].is_unique
assert reviews_clean["review_id"].notna().all()
assert reviews_clean["review_id"].is_unique

assert reviews_clean["product_id"].isin(
    products_clean["product_id"]
).all()

assert ingredients_clean["product_id"].isin(
    products_clean["product_id"]
).all()

assert concerns_clean["review_id"].isin(
    reviews_clean["review_id"]
).all()

assert ingredients_clean["ingredient_name_standardized"].notna().all()
assert concerns_clean["skin_concern_standardized"].notna().all()

assert not ingredients_clean.duplicated(
    ["product_id", "ingredient_position"]
).any()

assert not concerns_clean.duplicated(
    ["review_id", "skin_concern_standardized"]
).any()

# Save separately from the original collection files.
cleaned_folder = (
    PROJECT_ROOT
    / "data"
    / "cleaned"
    / "all_products"
)
cleaned_folder.mkdir(parents=True, exist_ok=True)

for name, table in cleaned_tables.items():
    file_path = cleaned_folder / f"{name}.csv"
    table.to_csv(file_path, index=False)

    saved = pd.read_csv(
        file_path,
        dtype={"product_id": str, "review_id": str, "sku": str}
    )

    assert len(saved) == len(table), f"Row count mismatch: {name}"
    assert saved.columns.tolist() == table.columns.tolist(), (
        f"Column mismatch: {name}"
    )

    print(f"{name}.csv: {len(saved):,} rows saved")

print("\nAll cleaned-table checks passed.")
print("Saved to:", cleaned_folder.resolve())